# YOLO-Modell-Finetuning

Das vortrainierte YOLO-Modell wird auf den gegebenen GTSRB-Datensatz angepasst (Finetuning).

Hierzu wird der Klassifikationskopf des Modells durch einen vollvernetzten Layer ersetzt oder ergänzt, dessen Anzahl an Ausgabeneuronen der Anzahl der Klassen im Datensatz entspricht. Die Gewichte des restlichen Netzwerks bleiben dabei unverändert (eingefroren), sodass ausschließlich der neue Layer trainiert wird, um die Trainingszeit gering zu halten.

Weitere Informationen über das Trainieren des YOLO-Modells sind auf dieser Seite zu finden:

- https://docs.ultralytics.com/tasks/classify/#train

## Installation und Importe

Es wird die `ultralytics`-Bibliothek installiert und notwendige Python-Module importiert.

In [ ]:
!pip install ultralytics -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 67.9 MB/s eta 0:00:00


In [ ]:
from sklearn.utils import shuffle
from ultralytics import YOLO
from collections import Counter
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import pickle
import os
import cv2
import yaml

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


## Konfiguration der Laufzeitumgebung

Das Skript erkennt automatisch, ob es in Google Colab oder lokal ausgeführt wird, und bindet bei Bedarf Google Drive ein.

Die Beständigkeit der Daten ist wichtig. Da Colab-Laufzeiten temporär sind, ermöglicht die Anbindung von Google Drive den Zugriff auf große Datensätze, ohne diese jedes Mal neu hochladen zu müssen.

In [ ]:
# Umgebung erkennen: Colab oder lokal
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
    BASE_PATH = '/content/drive/MyDrive/'  # ggf. an Projektordner anpassen
    print("Running in Google Colab – Google Drive wurde eingebunden.")
except ModuleNotFoundError:
    IN_COLAB = False
    BASE_PATH = './'  # lokales Arbeitsverzeichnis, ggf. anpassen
    print("Running locally – Google Drive wird nicht eingebunden.")

Mounted at /content/drive
Running in Google Colab – Google Drive wurde eingebunden.


## Laden des GTSRB-Datensatzes

Es werden die Pfade zu den Trainings- und Validierungsdaten definiert und die vorverarbeiteten Pickle-Dateien geladen.

Um ein Modell zu trainieren, werden zwei getrennte Datensätze benötigt: den **Trainingsdatensatz**, aus dem das Modell lernt, und den **Validierungsdatensatz**, um die Genauigkeit auf ungesehenen Daten objektiv zu beurteilen und eine Überanpassung (Overfitting) zu erkennen.

In [ ]:
# Zentrale Pfad-Konfiguration – abhängig von der Umgebung
if IN_COLAB:
    DATA_DIR = '/content/drive/MyDrive/Colab Notebooks/KI-thkoeln/praktikum_2/assets/GTSRB'
else:
    # Lokaler Pfad – bei Bedarf an eigene Ordnerstruktur anpassen
    DATA_DIR = os.path.join('assets', 'GTSRB')

In [ ]:
valid_dir = os.path.join(DATA_DIR, 'valid.p')
train_dir = os.path.join(DATA_DIR, 'train.p')

with open(valid_dir, mode='rb') as valid_dataset:
    valid = pickle.load(valid_dataset)
with open(train_dir, mode='rb') as train_dataset:
    train = pickle.load(train_dataset)

X_train, y_train = train["features"], train["labels"]
X_valid, y_valid = valid["features"], valid["labels"]

### Datensatz für YOLO vorbereiten

In diesem Schritt wird der GTSRB-Datensatz in eine Struktur transformiert, die mit dem Ultralytics YOLO-Framework kompatibel ist. Dies ist aus folgenden Gründen wichtig:

1.  YOLO (Classification) erwartet Bilder in Unterordnern, wobei jeder Ordnername der Klassen-ID entspricht.
2.  Es wird ein Dictionary (`gtsrb_classes_text`) erstellt, um den numerischen IDs verständliche Namen zuzuweisen (z.B. `1` zu `Speed limit (30km/h)`), was die spätere Interpretation der Ergebnisse erleichtert.
3.  Da die Daten ursprünglich in einem Python-Pickle-Format vorliegen, werden die Pixel-Arrays extrahiert und als `.png`-Dateien auf der Festplatte gespeichert, damit der YOLO-Dataloader sie effizient einlesen kann.
4.  Die YAML-Datei definiert, wo sich die Trainings- und Validierungsbilder befinden und wie die Klassen benannt sind.

In [ ]:
# Mapping der GTSRB Klassen-IDs zu Text-Labels
gtsrb_classes_text = {
    0: 'Speed limit (20km/h)',
    1: 'Speed limit (30km/h)',
    2: 'Speed limit (50km/h)',
    3: 'Speed limit (60km/h)',
    4: 'Speed limit (70km/h)',
    5: 'Speed limit (80km/h)',
    6: 'End of speed limit (80km/h)',
    7: 'Speed limit (100km/h)',
    8: 'Speed limit (120km/h)',
    9: 'No passing',
    10: 'No passing veh over 3.5t',
    11: 'Right-of-way at intersection',
    12: 'Priority road',
    13: 'Yield',
    14: 'Stop',
    15: 'No vehicles',
    16: 'Veh > 3.5t prohibited',
    17: 'No entry',
    18: 'General caution',
    19: 'Dangerous curve left',
    20: 'Dangerous curve right',
    21: 'Double curve',
    22: 'Bumpy road',
    23: 'Slippery road',
    24: 'Road narrows on the right',
    25: 'Road work',
    26: 'Traffic signals',
    27: 'Pedestrians',
    28: 'Children crossing',
    29: 'Bicycles crossing',
    30: 'Beware of ice/snow',
    31: 'Wild animals crossing',
    32: 'End speed + passing limits',
    33: 'Turn right ahead',
    34: 'Turn left ahead',
    35: 'Ahead only',
    36: 'Go straight or right',
    37: 'Go straight or left',
    38: 'Keep right',
    39: 'Keep left',
    40: 'Roundabout mandatory',
    41: 'End of no passing',
    42: 'End no passing veh > 3.5t'
}

# Anzahl der Klassen aus den Trainingslabels berechnen
num_classes_gtsrb = len(set(y_train))
print(f"Anzahl der Klassen im GTSRB-Datensatz: {num_classes_gtsrb}")

# Basisverzeichnis für den Datensatz definieren
base_dataset_dir = 'yolo_gtsrb_dataset'
os.makedirs(base_dataset_dir, exist_ok=True)

# Train- und Validierungsverzeichnisse sowie Klassenunterverzeichnisse erstellen
train_img_base_dir = os.path.join(base_dataset_dir, 'train')
val_img_base_dir = os.path.join(base_dataset_dir, 'val')

for i in range(num_classes_gtsrb):
    os.makedirs(os.path.join(train_img_base_dir, str(i)), exist_ok=True)
    os.makedirs(os.path.join(val_img_base_dir, str(i)), exist_ok=True)

print("Speichere Trainingsbilder...")
for i, (image, label) in enumerate(zip(X_train, y_train)):
    class_folder = os.path.join(train_img_base_dir, str(label))
    cv2.imwrite(os.path.join(class_folder, f'{i:05d}.png'), image)
print(f"{len(X_train)} Trainingsbilder gespeichert.")

print("Speichere Validierungsbilder...")
for i, (image, label) in enumerate(zip(X_valid, y_valid)):
    class_folder = os.path.join(val_img_base_dir, str(label))
    cv2.imwrite(os.path.join(class_folder, f'{i:05d}.png'), image)
print(f"{len(X_valid)} Validierungsbilder gespeichert.")

data_config = {
    'path': base_dataset_dir,
    'train': 'train',
    'val': 'val',
    'names': gtsrb_classes_text
}

data_yaml_path = os.path.join(base_dataset_dir, 'data.yaml')
with open(data_yaml_path, 'w') as file:
    yaml.dump(data_config, file, default_flow_style=False)

print(f"data.yaml erstellt unter {data_yaml_path}")

Anzahl der Klassen im GTSRB-Datensatz: 43
Speichere Trainingsbilder...
34799 Trainingsbilder gespeichert.
Speichere Validierungsbilder...
4410 Validierungsbilder gespeichert.
data.yaml erstellt unter yolo_gtsrb_dataset/data.yaml


## Finetuning des YOLO-Klassifizierungsmodells

Beim Finetuning wird das Wissen eines bereits vortrainierten Modells (`yolo26n-cls.pt`) genutzt.

Anstatt bei Null anzufangen, wird das Modell nur an die spezifischen Verkehrszeichen angepasst. Das spart massiv Zeit und Rechenleistung.

Durch den Parameter `freeze=True` (oder das Einfrieren der unteren Layer) wird verhindert, dass die bereits gelernten allgemeinen Merkmale überschrieben werden. Nur der neue Klassifikationskopf am Ende des Netzwerks wird trainiert.

Das angepasste Modell wird im Ordner `/content/runs/classify/YOLO_GTSRB_cls_finetuning/yolo26n-cls_finetuned/weights/best.pt` gespeichert und kann dort heruntergeladen werden.

In [ ]:
model = YOLO("yolo26n-cls.pt")

In [ ]:
print(model.model) # Details zum Model

In [ ]:
print("Starte Finetuning des YOLO-Modells...")

"""
TODO: Trainieren Sie das YOLO-Modell mit den GTSRB-Trainigsdatensatz (siehe https://docs.ultralytics.com/tasks/classify/#train)
"""

print("Finetuning abgeschlossen.")

In [ ]:
print(model.model) # Details zum Model

## Evaluierung des feinabgestimmten Modells

Nach dem Training muss objektiv gemessen werden, wie gut das Modell auf Daten performt, die während des Lernprozesses nicht gesehen wurden (der Validierungsdatensatz).

In [ ]:
# Explizites Setzen der Klassennamen im Modell-Objekt für die Visualisierung
model.model.names = gtsrb_classes_text

"""
TODO: Evaluiere Sie das feinabgestimmte Modell auf dem GTSRB-Validierungs-Set
"""

### Visualisierung der Ergebnisse

Es werden die Trainingskurven (Loss/Accuracy) und die Confusion Matrix angezeigt, um den Trainingsverlauf zu bewerten.

Ein sinkender Loss zeigt, dass das Modell lernt, während die Confusion Matrix genau aufzeigt, bei welchen Klassen das Modell noch Schwierigkeiten hat (z.B. ähnliche Schilder).

In [ ]:
train_results_dir = model.trainer.save_dir

"""
TODO: Visualisieren Sie den Trainingsverlauf (Loss & Accuracy Graphen). Nutzen Sie dafür ggf. die Ergebnisse aus 'train_results_dir'
"""

In [ ]:
val_results_dir = metrics.save_dir

"""
TODO: Visualisieren Sie die Ergebnisse der Evaluierung (Konfusionsmatrix). Nutzen Sie dafür ggf. die Ergebnisse aus 'val_results_dir'
"""

In [ ]:

"""
TODO: Geben Sie die Top 10 Klassen mir Ihrer Häufigkeit aus.

Zum Beispiel:

Top 10 erkannte Klassen:
loupe: 1439
analog_clock: 223
...

"""
